# DrugReview: uncertainty of the holistic annotation (score stream)

The annotation protocol (`v2_ef356fdf87`) records the holistic label of every review. Its class-score stream (`s_7c599892d6`) asks
the same model, under the same rubric, to **report its distribution over the five labels** in a separate, independent request
per review. This notebook evaluates that reported distribution as a measure of annotation uncertainty:

1. integrity of the inputs (corpus hashes, counts, partitions);
2. the distribution of normalized entropy, its discrete structure, and its dependence on the holistic class;
3. agreement between the reported label and the frozen holistic label, and the role of ties in the reported vector;
4. entropy bands on the test partition: agreement with the binned rating and error rates of the selected holistic models (rungs 2 and 4);
5. entropy as a ranking of model disagreement, pooled and within class, and the triage curve;
6. the human-reference audit: entropy against the annotators' disagreements and their uncertainty flags;
7. repeatability of the score stream on the 2,000-review repeated subset;
8. entropy by aspect configuration;
9. entropy within the holistic class;
10. Table 3 of the manuscript, as printed there.

Every table is written to `outputs/` as CSV; `outputs/summary.json` holds the headline numbers. Nothing in the release
folder is modified. Conventions follow the manuscript: normalized entropy $H=-\frac{1}{\log 5}\sum_c p_c\log p_c$ of the
reported vector; the rating is binned 1--2, 3--4, 5--6, 7--8, 9--10; the test partition is the narrative-group split of `00_split`.

## 0. Setup: paths, constants, helpers

In [1]:
import os, json, glob, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, cohen_kappa_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

# ROOT is the folder that holds 00_labelling/, 00_split/, 02_4ominiLabel/, ... (Drive: NLP_Projects/03_DrugReview; local: "01. Colab backup").
if os.path.isdir("/content") and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import os
from pathlib import Path

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents,
                 Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

PROJECT_ROOT = resolve_project_root()

ROOT = PROJECT_ROOT
OUT  = Path(os.environ.get("DRUGREVIEW_ENTROPY_OUT", ROOT / "00_entropy" / "outputs")); OUT.mkdir(parents=True, exist_ok=True)
FIG  = OUT / "figures"; FIG.mkdir(exist_ok=True)

PROTOCOL_ID, SCORES_ID = "v2_ef356fdf87", "s_7c599892d6"
REL   = ROOT / "00_labelling" / "output_v2" / PROTOCOL_ID
CORPUS_MD5  = "2eb1f53e0ee6c19c63be3194f90f23af"         # frozen corpus (protocol.json)
RELEASE_MD5 = "c22f6c05633214014db47c692e045ea1"         # drugscom_plus_ai_labels_<id>.csv
CLASSES = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
LOWER   = [c.lower() for c in CLASSES]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
RUNS = {"04": "04_Holistic_all_aspects/outputs/aspect_auxiliary_v2_3class"}

def entropy_norm(P):
    """Normalized Shannon entropy of each row of P (rows need not be normalized)."""
    P = np.asarray(P, float); P = P / P.sum(1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        return -(np.where(P > 0, P * np.log(P), 0)).sum(1) / np.log(len(CLASSES))

def bin5(r):
    return "VERY_NEGATIVE" if r <= 2 else "NEGATIVE" if r <= 4 else "NEUTRAL" if r <= 6 else "POSITIVE" if r <= 8 else "VERY_POSITIVE"

def md5(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def safe_auc(y, s):
    y = np.asarray(y).astype(int); return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else np.nan

def save(df, name, **kw):
    df.to_csv(OUT / name, **kw); print(f"written: outputs/{name}")

SUMMARY = {"protocol_id": PROTOCOL_ID, "scores_id": SCORES_ID}
print("ROOT:", ROOT); print("OUT :", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/NLP_Projects/03_DrugReview
OUT : /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_entropy/outputs


## 1. Inputs and integrity

The reported vector comes from `scores/drugscom_holistic_scores_<SCORES_ID>.csv` (one row per review, keyed by `uniqueID`);
the frozen holistic label, the aspect states and the corpus columns from the release file. The two are joined on `uniqueID`,
and the join is checked: the same 102,061 reviews, and every score-stream answer carries the hash of the text loaded now.

In [2]:
rel_path = REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv"
sc_path  = REL / "scores" / f"drugscom_holistic_scores_{SCORES_ID}.csv"
assert rel_path.exists() and sc_path.exists(), "release or scores file missing"
print("release md5 matches the frozen value:", md5(rel_path) == RELEASE_MD5)

use = ["uniqueID", "rating", "ai_sent5_hard"] + [f"ai_{a}" for a in ASPECTS]
rel = pd.read_csv(rel_path, usecols=lambda c: c in use)
keep = ["uniqueID", "scores_status", "scores_label", "scores_label_is_argmax", "scores_entropy_norm", "scores_input_hash"] + [f"ai_s_{c}" for c in LOWER] + [f"ai_q_{c}" for c in LOWER]
sc  = pd.read_csv(sc_path, usecols=lambda c: c in keep)
assert sc.uniqueID.is_unique and rel.uniqueID.is_unique and set(sc.uniqueID) == set(rel.uniqueID), "id sets differ"
d = rel.merge(sc, on="uniqueID", validate="one_to_one")

# the score stream's answers must belong to the text that the corpus holds now
corpus_path = ROOT / "00_labelling" / "output" / "drugscom_labelling_wcge50_ok_dedup_text_cond_ingCLEAN.csv"
corpus = pd.read_csv(corpus_path, usecols=["uniqueID", "review", "ingredient_norm", "condition_norm"])
print("corpus md5 matches the frozen value:", md5(corpus_path) == CORPUS_MD5)
def payload(r):
    return json.dumps({"review": str(r.review).strip(), "ingredient": "" if pd.isna(r.ingredient_norm) else str(r.ingredient_norm),
                       "condition": "" if pd.isna(r.condition_norm) else str(r.condition_norm)}, ensure_ascii=False)
HASH = {u: hashlib.sha256(payload(r).encode("utf-8")).hexdigest()[:16] for u, r in zip(corpus.uniqueID, corpus.itertuples())}
hash_ok = (d.scores_input_hash == d.uniqueID.map(HASH)).mean()
print(f"score-stream answers with the corpus hash: {hash_ok:.4f} | status: {d.scores_status.value_counts().to_dict()}")
assert hash_ok == 1.0 and (d.scores_status == "OK").all()

# split
split = pd.read_csv(ROOT / "00_split" / "data" / "DrugReview_split.csv")
d = d.merge(split[["uniqueID", "narrative_group", "split", "is_human_annotated"]], on="uniqueID", validate="one_to_one")
print("partitions:", d.split.value_counts().to_dict(), "| human-annotated:", int(d.is_human_annotated.sum()))
assert len(d) == 102061 and (d.split == "test").sum() == 20298

# the reported vector, its entropy, and its ties
Sm = d[[f"ai_s_{c}" for c in LOWER]].to_numpy(float)                 # reported scores, as returned
d["H"] = entropy_norm(Sm); assert np.allclose(d.H, d.scores_entropy_norm)
top = np.isclose(Sm, Sm.max(1)[:, None]); d["n_top"] = top.sum(1); d["tie"] = d.n_top > 1
d["rating5"] = d.rating.map(bin5)
d["frozen_id"] = d.ai_sent5_hard.map(CLASSES.index); d["rep_id"] = d.scores_label.map(CLASSES.index)
print(f"reviews: {len(d):,} | reported label == argmax: {d.scores_label_is_argmax.astype(bool).mean():.4f} | top ties: {d.tie.mean():.4f}")
SUMMARY.update(n_reviews=int(len(d)), n_test=int((d.split == "test").sum()), tie_share=float(d.tie.mean()))

release md5 matches the frozen value: True
corpus md5 matches the frozen value: True
score-stream answers with the corpus hash: 1.0000 | status: {'OK': 102061}
partitions: {'train': 60950, 'test': 20298, 'val': 20256, 'audit': 557} | human-annotated: 500
reviews: 102,061 | reported label == argmax: 0.9980 | top ties: 0.1394


## 2. The distribution of normalized entropy

The reported vector is diffuse for most reviews and takes few distinct values, because the model reports in tenths and
reuses a handful of stereotyped vectors. Both facts matter for how entropy can be used: bands rather than a continuum, and
cut-points at the steps the reported scale actually takes.

In [3]:
qs = [0, .05, .10, .25, .50, .75, .90, .95, 1.0]
dist = d.H.quantile(qs); dist.index = ["min"] + [f"q{int(q*100):02d}" for q in qs[1:-1]] + ["max"]
dist = pd.concat([pd.Series({"mean": d.H.mean(), "SD": d.H.std()}), dist])
print("normalized entropy of the reported vector, full corpus\n", dist.round(3).to_string()); save(dist.round(4).to_frame("H"), "entropy_distribution_quantiles.csv", index_label="statistic")

edges = [-1e-9, 0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]; labels = ["= 0", "(0, 0.2]", "(0.2, 0.4]", "(0.4, 0.5]", "(0.5, 0.6]", "(0.6, 0.8]", "(0.8, 1]"]
bands = (pd.cut(d.H, edges, labels=labels).value_counts(normalize=True).reindex(labels) * 100).round(1).to_frame("share_pct")
print("\nshare of reviews per band (%)\n", bands.to_string()); save(bands, "entropy_distribution_bands.csv", index_label="band")

# the discrete structure of the reported vector
Hr = d.H.round(3); pat = pd.Series(["/".join(f"{v:g}" for v in sorted(r, reverse=True) if v > 0) for r in Sm], index=d.index)
vals = Hr.value_counts(normalize=True).head(15) * 100
steps = pd.DataFrame({"H": vals.index, "share_pct": vals.values.round(1), "typical_vector_sorted": [pat[Hr == v].value_counts().index[0] for v in vals.index]})
print(f"\ndistinct entropy values: {Hr.nunique()}; the 12 most frequent cover {Hr.value_counts(normalize=True).head(12).sum():.1%}")
print(steps.to_string(index=False)); save(steps, "entropy_discrete_steps.csv", index=False)
print("\nclasses with non-zero reported mass (% of reviews):", pd.Series((Sm > 0).sum(1)).value_counts(normalize=True).sort_index().mul(100).round(1).to_dict())

# dependence on the frozen holistic class
byc = d.groupby("ai_sent5_hard").agg(n=("H", "size"), H_median=("H", "median"), H_q25=("H", lambda x: x.quantile(.25)), H_q75=("H", lambda x: x.quantile(.75)),
                                    H_gt_0_5_pct=("H", lambda x: 100 * (x > .5).mean()), tie_pct=("tie", lambda x: 100 * x.mean())).reindex(CLASSES).round(3)
print("\nby frozen holistic class\n", byc.to_string()); save(byc, "entropy_by_frozen_class.csv", index_label="frozen_class")
for c in CLASSES:
    m = d.ai_sent5_hard == c; lab = pd.Series(["/".join(f"{v:g}" for v in r) for r in Sm[m.values]]).value_counts(normalize=True).head(3)
    print(f"  {c:14s} most common reported vectors (VN/N/NEU/P/VP): " + "; ".join(f"{k} ({v:.0%})" for k, v in lab.items()))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3.2), dpi=150)
ax.hist(d.H, bins=50, range=(0, 1), color="#4a6fa5"); ax.set_xlabel("normalized entropy $H$ of the reported vector"); ax.set_ylabel("reviews"); ax.axvline(0.5, ls="--", lw=.8, color="k")
fig.tight_layout(); fig.savefig(FIG / "entropy_distribution.png"); plt.close(fig); print("figure: outputs/figures/entropy_distribution.png")
SUMMARY.update(H_median=float(d.H.median()), H_q25=float(d.H.quantile(.25)), H_q75=float(d.H.quantile(.75)), H_share_gt_0_5=float((d.H > .5).mean()), H_distinct_values=int(Hr.nunique()))

normalized entropy of the reported vector, full corpus
 mean    0.569
SD      0.212
min     0.000
q05     0.311
q10     0.311
q25     0.418
q50     0.558
q75     0.795
q90     0.914
q95     0.914
max     0.967
written: outputs/entropy_distribution_quantiles.csv

share of reviews per band (%)
             share_pct
H                    
= 0               0.0
(0, 0.2]          0.0
(0.2, 0.4]       23.0
(0.4, 0.5]       19.9
(0.5, 0.6]       20.5
(0.6, 0.8]       23.5
(0.8, 1]         13.1
written: outputs/entropy_distribution_bands.csv

distinct entropy values: 41; the 12 most frequent cover 91.3%
    H  share_pct    typical_vector_sorted
0.311       18.4                  0.8/0.2
0.418       15.3                  0.6/0.4
0.795       12.2          0.4/0.3/0.2/0.1
0.558       10.6              0.6/0.3/0.1
0.742        6.6          0.4/0.4/0.1/0.1
0.914        5.7      0.4/0.2/0.2/0.1/0.1
0.498        4.2              0.7/0.2/0.1
0.586        4.1              0.5/0.4/0.1
0.935        3.9   

The frozen holistic label is the one recorded by the annotation protocol; the reported label is the label returned with the independently elicited score vector. Agreement below uses that returned label, while argmax consistency and ties are reported separately.


In [4]:
agree = (d.scores_label == d.ai_sent5_hard)
qwk = cohen_kappa_score(d.scores_label, d.ai_sent5_hard, labels=CLASSES, weights="quadratic")
conf = pd.crosstab(d.ai_sent5_hard, d.scores_label).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
print(f"reported label vs frozen label: agreement {agree.mean():.4f}, quadratic kappa {qwk:.4f}")
print("rows = frozen, columns = reported\n", conf.to_string()); save(conf, "labels_confusion.csv", index_label="frozen\\reported")

fi = d.frozen_id.to_numpy(); frozen_in_top = top[np.arange(len(d)), fi]; tied = d.tie.to_numpy()
sets = pd.Series(["+".join(CLASSES[i] for i in np.where(top[k])[0]) for k in np.where(tied)[0]]).value_counts()
ties = pd.DataFrame({"tied_classes": sets.index, "n": sets.values, "share_of_ties_pct": (sets.values / tied.sum() * 100).round(1)})
print("\ntop ties by class pair\n", ties.to_string(index=False)); save(ties, "labels_tie_pairs.csv", index=False)

dis = ~agree.to_numpy()
rows = {"agreement": agree.mean(), "qwk": qwk, "tie_share": tied.mean(),
        "frozen_among_top_classes_overall": frozen_in_top.mean(), "frozen_among_tied_classes": frozen_in_top[tied].mean(), "agreement_untied": agree[~tied].mean(),
        "tiebreak_picks_frozen": agree[tied].mean(), "n_disagreements": int(dis.sum()), "disagreements_forced_tiebreak": int((dis & tied).sum()),
        "disagreements_untied": int((dis & ~tied).sum()), "frozen_is_runner_up_in_untied_disagreements": float((np.argsort(-Sm, 1)[:, 1] == fi)[dis & ~tied].mean())}
ext = {"VERY_NEGATIVE": 2, "NEGATIVE": 1, "NEUTRAL": 0, "POSITIVE": 1, "VERY_POSITIVE": 2}
mv = d.loc[dis, ["ai_sent5_hard", "scores_label"]]; rows["disagreements_toward_more_extreme"] = float((mv.scores_label.map(ext) > mv.ai_sent5_hard.map(ext)).mean())
two = pd.Series(rows); print("\n", two.round(4).to_string()); save(two.to_frame("value"), "labels_summary.csv", index_label="quantity")
moves = (mv.ai_sent5_hard + " -> " + mv.scores_label).value_counts().head(8); print("\nmost frequent moves (frozen -> reported)\n", moves.to_string()); save(moves.to_frame("n"), "labels_moves.csv", index_label="move")
marg = pd.DataFrame({"frozen_pct": d.ai_sent5_hard.value_counts(normalize=True), "reported_pct": d.scores_label.value_counts(normalize=True), "rating5_pct": d.rating5.value_counts(normalize=True)}).reindex(CLASSES).mul(100).round(1)
print("\nmarginals (%)\n", marg.to_string()); save(marg, "labels_marginals.csv", index_label="class")
SUMMARY.update(agreement_reported_vs_frozen=float(agree.mean()), qwk_reported_vs_frozen=float(qwk), frozen_is_maximizer_of_reported=float(frozen_in_top.mean()),
               disagreements_forced_tiebreak_share=float((dis & tied).sum() / dis.sum()))

reported label vs frozen label: agreement 0.7511, quadratic kappa 0.9341
rows = frozen, columns = reported
 scores_label   VERY_NEGATIVE  NEGATIVE  NEUTRAL  POSITIVE  VERY_POSITIVE
ai_sent5_hard                                                           
VERY_NEGATIVE          18469         0        0         8             10
NEGATIVE               11516      5230       23       326             41
NEUTRAL                   52      2216      532       468              9
POSITIVE                  76      2435      325     21576           6350
VERY_POSITIVE              3         0        0      1547          30849
written: outputs/labels_confusion.csv

top ties by class pair
           tied_classes    n  share_of_ties_pct
VERY_NEGATIVE+NEGATIVE 6245               43.9
POSITIVE+VERY_POSITIVE 3735               26.3
     NEGATIVE+POSITIVE 3623               25.5
      NEUTRAL+POSITIVE  610                4.3
      NEGATIVE+NEUTRAL    6                0.0
NEGATIVE+VERY_POSITIVE    6         

## 4. Entropy bands on the test partition: rating agreement and model error

Entropy is read in bands on the test partition ($n=20{,}298$). The bands are cut at the steps of the reported scale
(0.311, 0.418/0.498, 0.558/0.586, 0.640/0.742, 0.795 and above), so that no band boundary falls on a tie. For each band:
its share of the partition, the share labeled neutral, the agreement between the binned rating and the holistic label, and
the error rate of the selected ALBERT and BioBERT models of rung 2 (holistic only) and rung 4 (holistic + aspect heads).

In [5]:
r2 = pd.read_csv(ROOT / "02_4ominiLabel" / "outputs" / "dl_models_predictions.csv", usecols=lambda c: c in ("uniqueID", "true_label_id", "albert_pred_id", "biobert_pred_id"))
r4a = pd.read_csv(ROOT / RUNS["04"] / "albert" / "albert_test_predictions.csv", usecols=["uniqueID", "overall_true_id", "overall_pred_id"]).rename(columns={"overall_pred_id": "r4_albert_pred_id"})
r4b = pd.read_csv(ROOT / RUNS["04"] / "biobert" / "biobert_test_predictions.csv", usecols=["uniqueID", "overall_pred_id"]).rename(columns={"overall_pred_id": "r4_biobert_pred_id"})
t = d[d.split == "test"].merge(r2, on="uniqueID", validate="one_to_one").merge(r4a, on="uniqueID", validate="one_to_one").merge(r4b, on="uniqueID", validate="one_to_one")
assert (t.true_label_id == t.frozen_id).all() and (t.overall_true_id == t.frozen_id).all(), "prediction files do not carry the frozen label"
MODELS = {k: v for k, v in {"r2_albert": "albert_pred_id", "r2_biobert": "biobert_pred_id", "r4_albert": "r4_albert_pred_id", "r4_biobert": "r4_biobert_pred_id"}.items() if v in t.columns}
for k, v in MODELS.items(): t[f"dis_{k}"] = (t[v] != t.frozen_id).astype(int)
print(f"test partition: {len(t):,} reviews | models: {list(MODELS)}")

EDGES, LAB = [-1e-9, .35, .52, .60, .77, 1.0], ["<= 0.311", "0.418-0.498", "0.558-0.586", "0.640-0.742", ">= 0.795"]
b = pd.cut(t.H, EDGES, labels=LAB); g = t.groupby(b, observed=False)
tb = pd.DataFrame({"share_pct": g.size() / len(t) * 100, "neutral_pct": g.apply(lambda s: 100 * (s.ai_sent5_hard == "NEUTRAL").mean()),
                   "rating_agreement_pct": g.apply(lambda s: 100 * (s.rating5 == s.ai_sent5_hard).mean())})
for k in MODELS: tb[f"error_{k}_pct"] = g[f"dis_{k}"].mean() * 100
tb.loc["All"] = pd.Series({"share_pct": 100.0, "neutral_pct": 100 * (t.ai_sent5_hard == "NEUTRAL").mean(), "rating_agreement_pct": 100 * (t.rating5 == t.ai_sent5_hard).mean(), **{f"error_{k}_pct": 100 * t[f"dis_{k}"].mean() for k in MODELS}})
tb = tb.round(1); print("\nentropy bands, test partition\n", tb.to_string()); save(tb, "bands_test.csv", index_label="band")

rb = pd.DataFrame({"share_pct": g.size() / len(t) * 100, "reported_label_eq_rating5_pct": g.apply(lambda s: 100 * (s.scores_label == s.rating5).mean()),
                   "frozen_label_eq_rating5_pct": g.apply(lambda s: 100 * (s.ai_sent5_hard == s.rating5).mean()), "reported_eq_frozen_pct": g.apply(lambda s: 100 * (s.scores_label == s.ai_sent5_hard).mean())}).round(1)
print("\nagreement with the binned rating by entropy band\n", rb.to_string()); save(rb, "bands_test_rating.csv", index_label="band")
print(f"\nSpearman(H, reported label == rating5) = {spearmanr(t.H, (t.scores_label == t.rating5).astype(int)).correlation:+.3f}; "
      f"Spearman(H, frozen label == rating5) = {spearmanr(t.H, (t.ai_sent5_hard == t.rating5).astype(int)).correlation:+.3f}")
SUMMARY.update(band_low_error_r2_albert_pct=float(tb.loc["<= 0.311", "error_r2_albert_pct"]) if "error_r2_albert_pct" in tb else None,
               band_top_error_r2_albert_pct=float(tb.loc[">= 0.795", "error_r2_albert_pct"]) if "error_r2_albert_pct" in tb else None)

test partition: 20,298 reviews | models: ['r2_albert', 'r2_biobert', 'r4_albert', 'r4_biobert']

entropy bands, test partition
              share_pct  neutral_pct  rating_agreement_pct  error_r2_albert_pct  error_r2_biobert_pct  error_r4_albert_pct  error_r4_biobert_pct
H                                                                                                                                              
<= 0.311          21.9          0.0                  93.6                  5.9                   6.2                  4.5                   5.0
0.418-0.498       21.3          0.3                  68.6                 23.6                  24.2                 23.6                  23.0
0.558-0.586       20.7          0.2                  51.3                 24.7                  27.1                 25.2                  26.5
0.640-0.742       11.3          1.8                  32.8                 21.0                  27.4                 23.3                  24.1
>= 0.795

## 5. Entropy as a ranking of model disagreement: pooled, within class, and the triage curve

For each selected model, the AUC with which $H$ ranks the reviews on which the model disagrees with the annotation, pooled
and within each holistic class (the within-class mean weights the classes by size). The triage curve then asks the
operational question: to hold a given share of the disagreements in the routed set, what share of the partition must be
routed, at each step of the reported scale.

In [6]:
rows = []
for c in CLASSES:
    s = t[t.ai_sent5_hard == c]; r = {"class": c, "n": len(s)}
    for k in MODELS: r[f"disagree_{k}_pct"] = round(100 * s[f"dis_{k}"].mean(), 1); r[f"auc_{k}"] = round(safe_auc(s[f"dis_{k}"], s.H), 3)
    rows.append(r)
w = t.ai_sent5_hard.value_counts(normalize=True); r = {"class": "All (pooled)", "n": len(t)}
for k in MODELS: r[f"disagree_{k}_pct"] = round(100 * t[f"dis_{k}"].mean(), 1); r[f"auc_{k}"] = round(safe_auc(t[f"dis_{k}"], t.H), 3)
rows.append(r); r = {"class": "All (size-weighted within-class)", "n": len(t)}
for k in MODELS: r[f"auc_{k}"] = round(sum(w[c] * safe_auc(t[t.ai_sent5_hard == c][f"dis_{k}"], t[t.ai_sent5_hard == c].H) for c in CLASSES), 3)
rows.append(r); wc = pd.DataFrame(rows)
print("AUC of H for model disagreement\n", wc.to_string(index=False)); save(wc, "auc_within_class.csv", index=False)

key_model = "r2_albert" if "r2_albert" in MODELS else list(MODELS)[0]; H = t.H.to_numpy(); dis = t[f"dis_{key_model}"].to_numpy(); out = []
for thr in sorted(set(np.round(H, 3))):
    m = H >= thr
    if 0 < m.mean() < 1: out.append({"threshold_H_ge": thr, "routed_pct": round(100 * m.mean(), 1), "disagreements_captured_pct": round(100 * dis[m].sum() / dis.sum(), 1), "disagreement_rate_in_routed_pct": round(100 * dis[m].mean(), 1), "disagreement_rate_in_kept_pct": round(100 * dis[~m].mean(), 1)})
tr = pd.DataFrame(out); print(f"\nTRIAGE ({key_model}) at every step of the reported scale\n", tr.to_string(index=False)); save(tr, "triage_curve.csv", index=False)
for thr in (.558, .795):
    m = t.H >= thr; print(f"H >= {thr}: {m.mean():.1%} of the partition holds {t.loc[m, f'dis_{key_model}'].sum() / t[f'dis_{key_model}'].sum():.1%} of {key_model} disagreements")
SUMMARY.update({f"auc_pooled_{k}": float(wc.loc[wc["class"] == "All (pooled)", f"auc_{k}"].iloc[0]) for k in MODELS})
SUMMARY.update({f"auc_within_{k}": float(wc.loc[wc["class"].str.startswith("All (size"), f"auc_{k}"].iloc[0]) for k in MODELS})

AUC of H for model disagreement
                            class     n  disagree_r2_albert_pct  auc_r2_albert  disagree_r2_biobert_pct  auc_r2_biobert  disagree_r4_albert_pct  auc_r4_albert  disagree_r4_biobert_pct  auc_r4_biobert
                   VERY_NEGATIVE  3653                    14.5          0.600                     14.8           0.603                    14.6          0.603                     15.1           0.602
                        NEGATIVE  3461                    29.0          0.520                     32.6           0.500                    29.0          0.531                     30.3           0.507
                         NEUTRAL   637                    61.7          0.517                     60.9           0.496                    58.7          0.508                     71.0           0.517
                        POSITIVE  6090                    21.0          0.468                     28.6           0.492                    25.6          0.458              

## 6. Human-reference audit: entropy against the annotators

On the 500 human-annotated reviews, does $H$ separate (a) the reviews on which the two annotators disagree with each other,
(b) those on which the holistic annotation, and the reported label, disagree with each annotator, and (c) those an annotator
flagged as uncertain? The annotators' overall labels are read from the two workbooks, keyed to `uniqueID` through the sample
key exactly as the manuscript's scripts do (`source_row_id` indexes the release file). The low/high split uses the literal cutoff $H \geq 0.558$, preserving the original numerical analysis.

In [7]:
key = pd.read_csv(ROOT / "00_human_validation" / "subset" / "human_validation_sample_key_500.csv")
rel_full_ids = pd.read_csv(rel_path, usecols=["uniqueID"])
key["uniqueID"] = rel_full_ids.iloc[key.source_row_id.values]["uniqueID"].values
ann = {n: pd.read_excel(ROOT / "00_human_validation" / "outputs" / f"human_validation_annotator{n}_500.xlsm", sheet_name="Annotation") for n in (1, 2)}
h = key[["sample_id", "uniqueID"]].merge(d, on="uniqueID", validate="one_to_one")
for n in (1, 2):
    a = ann[n][["sample_id", "human_overall_sentiment", "human_uncertain"]].rename(columns={"human_overall_sentiment": f"A{n}", "human_uncertain": f"unc{n}"})
    h = h.merge(a, on="sample_id", validate="one_to_one")
for c in ("A1", "A2"): h[c] = h[c].astype(str).str.strip().str.upper()
for c in ("unc1", "unc2"): h[c] = h[c].astype(str).str.strip().str.upper().eq("YES").astype(int)
assert len(h) == 500 and h.is_human_annotated.all()
h["dis_A1_A2"] = (h.A1 != h.A2).astype(int); h["dis_LLM_A1"] = (h.ai_sent5_hard != h.A1).astype(int); h["dis_LLM_A2"] = (h.ai_sent5_hard != h.A2).astype(int)
h["dis_REP_A1"] = (h.scores_label != h.A1).astype(int); h["dis_REP_A2"] = (h.scores_label != h.A2).astype(int)
print(f"human subset: {len(h)} | A1 vs A2 agreement {1 - h.dis_A1_A2.mean():.3f} | frozen label vs A1 {1 - h.dis_LLM_A1.mean():.3f}, vs A2 {1 - h.dis_LLM_A2.mean():.3f} | reported label vs A1 {1 - h.dis_REP_A1.mean():.3f}, vs A2 {1 - h.dis_REP_A2.mean():.3f}")

SPLIT = .558; lo, hi = h.H < SPLIT, h.H >= SPLIT; rows = []
for tgt in ("dis_A1_A2", "dis_LLM_A1", "dis_LLM_A2", "dis_REP_A1", "dis_REP_A2", "unc1", "unc2"):
    rows.append({"target": tgt, "n_positive": int(h[tgt].sum()), "auc": round(safe_auc(h[tgt], h.H), 3), "rate_low_H_pct": round(100 * h.loc[lo, tgt].mean(), 1),
                 "rate_high_H_pct": round(100 * h.loc[hi, tgt].mean(), 1), "share_high_H_pct": round(100 * hi.mean(), 1), "spearman": round(spearmanr(h.H, h[tgt]).correlation, 3)})
aud = pd.DataFrame(rows); print(f"\nentropy vs disagreement / uncertainty flags on the 500 audited reviews (low/high H split at {SPLIT})\n", aud.to_string(index=False))
save(aud, "audit_entropy_vs_disagreement.csv", index=False)
rows = []
for c in CLASSES:
    s = h[h.ai_sent5_hard == c]
    rows.append({"class": c, "n": len(s), "A1_A2_disagree_pct": round(100 * s.dis_A1_A2.mean(), 1), "auc_A1A2": round(safe_auc(s.dis_A1_A2, s.H), 3),
                 "LLM_A2_disagree_pct": round(100 * s.dis_LLM_A2.mean(), 1), "auc_LLM_A2": round(safe_auc(s.dis_LLM_A2, s.H), 3), "LLM_A1_disagree_pct": round(100 * s.dis_LLM_A1.mean(), 1), "auc_LLM_A1": round(safe_auc(s.dis_LLM_A1, s.H), 3)})
audc = pd.DataFrame(rows); print("\nwithin frozen class\n", audc.to_string(index=False)); save(audc, "audit_entropy_within_class.csv", index=False)
SUMMARY.update(audit_auc_A1A2=float(aud.query("target=='dis_A1_A2'").auc.iloc[0]), audit_auc_LLM_A2=float(aud.query("target=='dis_LLM_A2'").auc.iloc[0]), audit_auc_LLM_A1=float(aud.query("target=='dis_LLM_A1'").auc.iloc[0]))

human subset: 500 | A1 vs A2 agreement 0.444 | frozen label vs A1 0.434, vs A2 0.750 | reported label vs A1 0.348, vs A2 0.642

entropy vs disagreement / uncertainty flags on the 500 audited reviews (low/high H split at 0.558)
     target  n_positive   auc  rate_low_H_pct  rate_high_H_pct  share_high_H_pct  spearman
 dis_A1_A2         278 0.563            53.7             58.1              43.0     0.109
dis_LLM_A1         283 0.519            57.2             55.8              43.0     0.033
dis_LLM_A2         125 0.627            20.4             31.2              43.0     0.191
dis_REP_A1         326 0.588            63.9             67.0              43.0     0.147
dis_REP_A2         179 0.694            25.6             49.3              43.0     0.324
      unc1          20 0.538             3.2              5.1              43.0     0.026
      unc2           8 0.737             0.4              3.3              43.0     0.103
written: outputs/audit_entropy_vs_disagreement.csv



## 7. Repeatability of the score stream (2,000-review repeated subset)

The same 2,000 reviews as the repeat of the holistic and aspect streams were read a second time by the score stream. Label agreement, kappa, the $L_1$ distance between the two reported vectors, and the stability of the entropy
ranking and of the band membership are recomputed from the part files; the repeat table of the holistic and aspect streams (labels) is
shown beside them.

In [8]:
def latest_ok(stage, task):
    recs = {}
    for f in sorted(glob.glob(str(REL / stage / "parts" / "parsed_*.jsonl"))):
        for l in open(f, encoding="utf-8"):
            if l.strip():
                r = json.loads(l)
                if r["task"] == task and (r["custom_id"] not in recs or r["attempt"] >= recs[r["custom_id"]]["attempt"]): recs[r["custom_id"]] = r
    P = pd.DataFrame(recs.values()); P = P[P.status == "OK"].copy(); P["uniqueID"] = P.uniqueID.astype(int); return P.set_index("uniqueID")
p1 = latest_ok("scores_production", "holistic_scores"); p2 = latest_ok("scores_repeat", "holistic_scores_rep")
idx = p1.index.intersection(p2.index); pc = [f"p_{c}" for c in LOWER]
A = p1.loc[idx, pc].astype(float).to_numpy(); B = p2.loc[idx, pc].astype(float).to_numpy(); A /= A.sum(1, keepdims=True); B /= B.sum(1, keepdims=True)
H1, H2 = entropy_norm(A), entropy_norm(B); l1 = np.abs(A - B).sum(1); mx = np.abs(A - B).max(1)
t1 = np.isclose(A, A.max(1)[:, None]).sum(1) > 1; t2 = np.isclose(B, B.max(1)[:, None]).sum(1) > 1; flip = (p1.loc[idx, "label"] != p2.loc[idx, "label"]).to_numpy()
rep = {"n": int(len(idx)), "label_agreement": float(1 - flip.mean()), "kappa": float(cohen_kappa_score(p1.loc[idx, "label"], p2.loc[idx, "label"])),
       "quadratic_kappa": float(cohen_kappa_score(p1.loc[idx, "label"], p2.loc[idx, "label"], labels=CLASSES, weights="quadratic")),
       "L1_mean": float(l1.mean()), "L1_median": float(np.median(l1)), "maxabs_lt_0.05": float((mx < .05).mean()), "maxabs_ge_0.2": float((mx >= .2).mean()),
       "H_spearman": float(spearmanr(H1, H2).correlation), "H_absdiff_median": float(np.median(np.abs(H1 - H2))),
       "tie_run1": float(t1.mean()), "tie_run2": float(t2.mean()), "n_label_flips": int(flip.sum()), "flips_involving_a_tie": int((flip & (t1 | t2)).sum()),
       "flip_rate_untied_both": float(flip[~(t1 | t2)].mean()), "flip_rate_tied_either": float(flip[(t1 | t2)].mean()),
       "band_0.558_agreement": float(((H1 >= .558) == (H2 >= .558)).mean()), "band_0.795_agreement": float(((H1 >= .795) == (H2 >= .795)).mean())}
rep_s = pd.Series(rep); print("score stream, repeated subset\n", rep_s.round(4).to_string()); save(rep_s.to_frame("value"), "repeat_score_stream.csv", index_label="quantity")
digit = pd.read_csv(REL / "stability_repeated_subset.csv"); digit = digit[~digit.field.str.startswith("fused")]   # holistic label and aspect fields only
print("\nfrozen protocol repeat (labels), for comparison\n", digit.to_string(index=False))
SUMMARY.update(repeat_label_agreement=rep["label_agreement"], repeat_H_spearman=rep["H_spearman"])

score stream, repeated subset
 n                        2000.0000
label_agreement             0.9515
kappa                       0.9320
quadratic_kappa             0.9855
L1_mean                     0.0576
L1_median                   0.0000
maxabs_lt_0.05              0.7840
maxabs_ge_0.2               0.0420
H_spearman                  0.9609
H_absdiff_median            0.0000
tie_run1                    0.1430
tie_run2                    0.1365
n_label_flips              97.0000
flips_involving_a_tie      52.0000
flip_rate_untied_both       0.0271
flip_rate_tied_either       0.1534
band_0.558_agreement        0.9435
band_0.795_agreement        0.9765
written: outputs/repeat_score_stream.csv

frozen protocol repeat (labels), for comparison
                   field    n  agreement  kappa  quadratic kappa
   holistic (5 classes) 2000     0.9895 0.9859           0.9967
     efficacy state (5) 2000     0.9675 0.9375              NaN
efficacy vote (-1/0/+1) 2000     0.9685 0.9383          

## 8. Entropy by aspect configuration

Using the frozen aspect states, reviews are grouped by whether the narrative reports a benefit (efficacy favorable) and
whether it reports a harm or a cost (safety, burden or cost unfavorable). If the reported entropy is a property of the
task, it should be highest where the two pull against each other; the table within class checks that the configuration
effect is not the class effect.

In [9]:
eff = d.ai_efficacy == "POSITIVE"; bad = (d[[f"ai_{a}" for a in ("safety", "burden", "cost")]] == "NEGATIVE").any(axis=1)
grp = pd.Series(np.select([eff & bad, eff & ~bad, ~eff & bad], ["benefit and harm/cost", "benefit, no harm/cost", "no benefit, harm/cost"], "neither"), index=d.index)
asp = d.groupby(grp).agg(n=("H", "size"), H_median=("H", "median"), H_gt_0_5_pct=("H", lambda x: 100 * (x > .5).mean()), tie_pct=("tie", lambda x: 100 * x.mean()),
                          reported_eq_frozen_pct=("scores_label", lambda s: 100 * (s == d.loc[s.index, "ai_sent5_hard"]).mean())).round(3)
print(asp.to_string()); save(asp, "entropy_by_aspect_configuration.csv", index_label="aspect_configuration")
w = d.assign(cfg=grp).groupby(["ai_sent5_hard", "cfg"]).H.agg(["size", "median"]).unstack("cfg").round(3)
print("\nmedian entropy by frozen class x configuration\n", w["median"].reindex(CLASSES).to_string()); save(w, "entropy_by_class_x_configuration.csv")

                           n  H_median  H_gt_0_5_pct  tie_pct  reported_eq_frozen_pct
benefit and harm/cost  36668     0.640        64.667   15.368                  70.587
benefit, no harm/cost  28081     0.311        21.648    6.196                  88.800
neither                 8693     0.640        75.521   21.258                  65.869
no benefit, harm/cost  28619     0.563        76.523   17.478                  70.271
written: outputs/entropy_by_aspect_configuration.csv

median entropy by frozen class x configuration
 cfg            benefit and harm/cost  benefit, no harm/cost  neither  no benefit, harm/cost
ai_sent5_hard                                                                              
VERY_NEGATIVE                  0.558                  0.558    0.558                  0.558
NEGATIVE                       0.742                  0.795    0.742                  0.742
NEUTRAL                        0.935                  0.914    0.914                  0.935
POSITIVE

## 9. Entropy within the holistic class

$H$ follows the holistic class closely (section 2), so an association between $H$ and disagreement can come from the class mix alone. For the rating and for each human comparison, this section reports the AUC of $H$ pooled, the AUC of the class alone, and the AUC of $H$ within each class (size-weighted over classes where it is defined). The 500 human-verified reviews are a stratified sample, so their within-class AUC is the fairer reading; their 95% intervals come from 1,000 review-level bootstrap resamples. The section also writes the class mix of the 500 against the release.

In [10]:
# Entropy within the holistic class. H follows the class closely (section 2), so an association
# between H and disagreement can come from the class mix alone. For each target this cell reports the AUC of H pooled,
# the AUC of the class alone (each review scored by its class's disagreement rate), and the AUC of H within each class
# (size-weighted mean over classes where the AUC is defined). On the 500 human-verified reviews, which are a stratified
# sample, 95% intervals come from 1,000 review-level bootstrap resamples (seed 2026), as for the agreement statistics.
# H is rounded to 6 decimals before ranking: vectors with the same entropy (e.g. 0.8/0.2 and 0.2/0.8) differ in the last
# floating-point bits when recomputed, which would otherwise break genuine ties arbitrarily.
def _auc(y, s):
    y = np.asarray(y).astype(int); s = np.round(np.asarray(s, float), 6)
    return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else np.nan

def _within(df, target):
    per = {c: _auc(g[target], g.H) for c, g in df.groupby("ai_sent5_hard")}
    n = df.ai_sent5_hard.value_counts(); ok = [c for c in per if not np.isnan(per[c])]
    return float(sum(per[c] * n[c] for c in ok) / sum(n[c] for c in ok)), per

def _summary(df, target):
    cls_rate = df.ai_sent5_hard.map(df.groupby("ai_sent5_hard")[target].mean())
    w, per = _within(df, target)
    return {"n": len(df), "n_positive": int(df[target].sum()), "pooled_auc": _auc(df[target], df.H), "class_only_auc": _auc(df[target], cls_rate),
            "within_class_auc": w, **{f"within_{c.lower()}": per.get(c, np.nan) for c in CLASSES}}

def _boot(df, target, B=1000, seed=2026):
    rng = np.random.default_rng(seed); p, w = [], []
    for _ in range(B):
        s = df.iloc[rng.integers(0, len(df), len(df))]; p.append(_auc(s[target], s.H)); w.append(_within(s, target)[0])
    return {"pooled_lo": np.nanpercentile(p, 2.5), "pooled_hi": np.nanpercentile(p, 97.5), "within_lo": np.nanpercentile(w, 2.5), "within_hi": np.nanpercentile(w, 97.5)}

rows = []
dd = d.assign(dis_rating=(d.rating5 != d.ai_sent5_hard).astype(int))
for part, df in (("release", dd), ("test", dd[dd.split == "test"])):
    rows.append({"sample": part, "target": "rating_vs_holistic", **_summary(df, "dis_rating")})
for tgt in ("dis_LLM_A2", "dis_LLM_A1", "dis_A1_A2"):
    rows.append({"sample": "human_500", "target": tgt, **_summary(h, tgt), **_boot(h, tgt)})
wc = pd.DataFrame(rows).round(3); print(wc.to_string(index=False)); save(wc, "entropy_within_class_auc.csv", index=False)

mix = pd.DataFrame({"human_500_pct": h.ai_sent5_hard.value_counts(normalize=True), "release_pct": d.ai_sent5_hard.value_counts(normalize=True)}).reindex(CLASSES).mul(100).round(1)
print("\nholistic class mix of the 500 human-verified reviews against the release (%)\n", mix.to_string()); save(mix, "human_500_class_mix.csv", index_label="class")
SUMMARY.update(within_auc_rating_test=float(wc.query("sample=='test'").within_class_auc.iloc[0]),
               within_auc_LLM_A2=float(wc.query("target=='dis_LLM_A2'").within_class_auc.iloc[0]), within_auc_LLM_A1=float(wc.query("target=='dis_LLM_A1'").within_class_auc.iloc[0]))

   sample             target      n  n_positive  pooled_auc  class_only_auc  within_class_auc  within_very_negative  within_negative  within_neutral  within_positive  within_very_positive  pooled_lo  pooled_hi  within_lo  within_hi
  release rating_vs_holistic 102061       42731       0.754           0.807             0.539                 0.583            0.564           0.502            0.413                 0.624        NaN        NaN        NaN        NaN
     test rating_vs_holistic  20298        8499       0.755           0.804             0.546                 0.596            0.577           0.482            0.415                 0.629        NaN        NaN        NaN        NaN
human_500         dis_LLM_A2    500         125       0.628           0.591             0.600                 0.579            0.480           0.429            0.524                 0.763      0.570      0.678      0.545      0.650
human_500         dis_LLM_A1    500         283       0.519           0.

## 10. Table 3 of the manuscript

The three panels as printed in the manuscript. Panel A: the full release by holistic class, with an all-reviews row. Panel B: the test partition by band of $H$, with the share of reviews, the neutral share of the holistic label, the share with a tied top score, agreement of the binned rating with the holistic label, and agreement of the label returned with the scores with the holistic label. Panel C: the within-class AUC of $H$ for disagreement from section 9; the test-partition row gets a 95% interval from 1,000 narrative-group bootstrap resamples (seed 2025), the convention for intervals on the test partition.

In [11]:
# Panel A: full release by holistic class, and all reviews
q = lambda s: pd.Series({"n": len(s), "median_H": s.H.median(), "Q1": s.H.quantile(.25), "Q3": s.H.quantile(.75), "tied_top_pct": 100 * s.tie.mean()})
pb = pd.DataFrame({c: q(d[d.ai_sent5_hard == c]) for c in CLASSES}).T; pb.loc["All reviews"] = q(d)
pb = pb.round({"median_H": 3, "Q1": 3, "Q3": 3, "tied_top_pct": 1}); pb["n"] = pb["n"].astype(int)
print("Panel A: full release by holistic class\n", pb.to_string()); save(pb, "table3_panelA.csv", index_label="class")

# Panel B: test partition by band of H (the bands of section 4)
EDGES3, LAB3 = [-1e-9, .35, .52, .60, .77, 1.0], ["H<=0.35", "0.35<H<=0.52", "0.52<H<=0.60", "0.60<H<=0.77", "H>0.77"]
g3 = t.groupby(pd.cut(t.H, EDGES3, labels=LAB3), observed=False)
pa = pd.DataFrame({"share_pct": g3.size() / len(t) * 100,
                   "neutral_pct": g3.apply(lambda s: 100 * (s.ai_sent5_hard == "NEUTRAL").mean()),
                   "tied_top_pct": g3.tie.mean() * 100,
                   "rating_agreement_pct": g3.apply(lambda s: 100 * (s.rating5 == s.ai_sent5_hard).mean()),
                   "returned_eq_holistic_pct": g3.apply(lambda s: 100 * (s.scores_label == s.ai_sent5_hard).mean())})
pa.loc["All"] = [100.0, 100 * (t.ai_sent5_hard == "NEUTRAL").mean(), 100 * t.tie.mean(),
                 100 * (t.rating5 == t.ai_sent5_hard).mean(), 100 * (t.scores_label == t.ai_sent5_hard).mean()]
pa = pa.round(1); print("\nPanel B: test partition by band of H\n", pa.to_string()); save(pa, "table3_panelB.csv", index_label="band")

# Panel C: within-class AUC of H for disagreement (section 9); interval for the test-partition row by narrative group
tt = dd[dd.split == "test"].reset_index(drop=True)
codes = tt.narrative_group.astype("category").cat.codes.to_numpy(); G = codes.max() + 1
members = [np.flatnonzero(codes == k) for k in range(G)] if G < 5000 else pd.Series(np.arange(len(tt))).groupby(codes).apply(np.array).tolist()
rng = np.random.default_rng(2025); wb = []
for _ in range(1000):
    ix = np.concatenate([members[k] for k in rng.integers(0, G, G)]); wb.append(_within(tt.iloc[ix], "dis_rating")[0])
w_test = _within(tt, "dis_rating")[0]
pc = pd.DataFrame([{"comparison": "binned rating vs holistic label (test partition)", "reviews": len(tt), "within_class_auc": w_test,
                    "lo": np.percentile(wb, 2.5), "hi": np.percentile(wb, 97.5)}] +
                  [{"comparison": {"dis_LLM_A1": "LLM annotation vs annotator 1", "dis_LLM_A2": "LLM annotation vs annotator 2", "dis_A1_A2": "annotator 1 vs annotator 2"}[r.target],
                    "reviews": int(r.n), "within_class_auc": r.within_class_auc, "lo": r.within_lo, "hi": r.within_hi}
                   for r in wc.query("sample == 'human_500'").set_index("target").loc[["dis_LLM_A1", "dis_LLM_A2", "dis_A1_A2"]].reset_index().itertuples()]).round(3)
print("\nPanel C: within-class AUC of H for disagreement\n", pc.to_string(index=False)); save(pc, "table3_panelC.csv", index=False)

Panel A: full release by holistic class
                     n  median_H     Q1     Q3  tied_top_pct
VERY_NEGATIVE   18487     0.558  0.498  0.558           0.4
NEGATIVE        17136     0.742  0.742  0.845          39.5
NEUTRAL          3277     0.914  0.795  0.935          50.4
POSITIVE        30762     0.795  0.586  0.795          17.1
VERY_POSITIVE   32399     0.311  0.311  0.418           1.4
All reviews    102061     0.558  0.418  0.795          13.9
written: outputs/table3_panelA.csv

Panel B: test partition by band of H
               share_pct  neutral_pct  tied_top_pct  rating_agreement_pct  returned_eq_holistic_pct
H                                                                                                 
H<=0.35            21.9          0.0           0.0                  93.6                      99.7
0.35<H<=0.52       21.3          0.3           0.0                  68.6                      79.5
0.52<H<=0.60       20.7          0.2           0.0                  5

## 11. Summary

In [12]:
(OUT / "summary.json").write_text(json.dumps(SUMMARY, indent=2)); print(json.dumps(SUMMARY, indent=2))
print("\noutputs:", sorted(p.name for p in OUT.iterdir() if p.is_file()))

{
  "protocol_id": "v2_ef356fdf87",
  "scores_id": "s_7c599892d6",
  "n_reviews": 102061,
  "n_test": 20298,
  "tie_share": 0.13937743114411968,
  "H_median": 0.5579250481919705,
  "H_q25": 0.4181656600790517,
  "H_q75": 0.7952181416542043,
  "H_share_gt_0_5": 0.5707958965716581,
  "H_distinct_values": 41,
  "agreement_reported_vs_frozen": 0.751080236329254,
  "qwk_reported_vs_frozen": 0.9340538269049042,
  "frozen_is_maximizer_of_reported": 0.8343833589715954,
  "disagreements_forced_tiebreak_share": 0.39082857705176144,
  "band_low_error_r2_albert_pct": 5.9,
  "band_top_error_r2_albert_pct": 28.2,
  "auc_pooled_r2_albert": 0.617,
  "auc_pooled_r2_biobert": 0.641,
  "auc_pooled_r4_albert": 0.636,
  "auc_pooled_r4_biobert": 0.637,
  "auc_within_r2_albert": 0.593,
  "auc_within_r2_biobert": 0.592,
  "auc_within_r4_albert": 0.595,
  "auc_within_r4_biobert": 0.583,
  "audit_auc_A1A2": 0.563,
  "audit_auc_LLM_A2": 0.627,
  "audit_auc_LLM_A1": 0.519,
  "repeat_label_agreement": 0.9515,
  "r